In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [2]:
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
import os
import json
import pandas as pd
import traceback

In [4]:
API_KEY=os.getenv("GOOGLE_API_KEY")

In [5]:
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash", 
    temperature=0.2,
    google_api_key=API_KEY
)

In [6]:
llm

ChatGoogleGenerativeAI(profile={'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'image_url_inputs': True, 'image_tool_message': True, 'tool_choice': True}, model='models/gemini-2.5-flash', google_api_key=SecretStr('**********'), temperature=0.2, client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 0x000001E9635BFD70>, default_metadata=(), model_kwargs={})

In [7]:
from langchain_core.prompts import PromptTemplate
from langchain_classic.chains import LLMChain
from langchain_classic.chains import SequentialChain

# Generate first question

In [8]:
prompt = "Collect testimonial for my pizza restaurant"

In [9]:
first_question_prompt = PromptTemplate(
    input_variables=["business_prompt"],
    template="""
You are a friendly AI interviewer collecting a video testimonial.

Context:
{business_prompt}

Task:
Generate ONE engaging opening question to start the testimonial interview.

Rules:
- Keep it under 20 words
- Make it conversational
- Do NOT add explanations
- Only return the question

Question:
"""
)

In [10]:
first_question_chain = LLMChain(
    llm=llm,
    prompt=first_question_prompt,
    output_key="question"
)

C:\Users\Harsh_Kr\AppData\Local\Temp\ipykernel_7284\3756386027.py:1: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 1.0. Use `RunnableSequence, e.g., `prompt | llm`` instead.
  first_question_chain = LLMChain(


In [11]:
result = first_question_chain.invoke({
    "business_prompt": "Collect testimonial for my pizza restaurant"
})

print(result["question"])

What makes our pizza special to you?


# Generate follow up question for given above question and answer

In [12]:

followup_prompt = PromptTemplate(
    input_variables=["business_prompt", "conversation_history", "last_answer"],
    template="""
You are a friendly AI interviewer collecting a testimonial.

Business Context:
{business_prompt}

Conversation So Far:
{conversation_history}

User's Latest Answer:
{last_answer}

Task:
Generate ONE natural follow-up question based on the latest answer.

Rules:
- Do NOT repeat previous questions
- Keep it under 20 words
- Make it conversational
- Only return the question

Follow-up Question:
"""
)

In [13]:
followup_chain = LLMChain(
    llm=llm,
    prompt=followup_prompt,
    output_key="followup_question"
)

In [14]:
result = followup_chain.invoke({
    "business_prompt": "Collect testimonial for my pizza restaurant",
    "conversation_history": """
AI: What made you try our pizza for the first time?
User: My friend recommended it and the cheese looked amazing.
""",
    "last_answer": "My friend recommended it and the cheese looked amazing."
})

print(result["followup_question"])

And did the cheese live up to your expectations?


# Now full pipeline generation


In [15]:
MAX_QUESTIONS = 6

In [16]:
business_prompt = "Collect testimonial for my pizza restaurant"

In [17]:
conversation_history = ""
all_qa = []

In [18]:
first_result = first_question_chain.invoke({
    "business_prompt": business_prompt
})

In [19]:
current_question = first_result["question"]

In [20]:
print(f"AI: {current_question}")

AI: How does our pizza make your day better?


In [ ]:
for i in range(MAX_QUESTIONS):

    # ---- Simulated user input (replace later with speech-to-text) ----
    user_answer = input("User: ")

    # Store Q&A
    all_qa.append({
        "question": current_question,
        "answer": user_answer,
        "tone":"happy"
    })

    # Update conversation history
    conversation_history += f"\nAI: {current_question}\nUser: {user_answer}"

    # ---- STOP CONDITION ----
    if i == MAX_QUESTIONS - 1:
        print("\nInterview completed 🎬")
        break

    # ---- GENERATE FOLLOW-UP ----
    followup_result = followup_chain.invoke({
        "business_prompt": business_prompt,
        "conversation_history": conversation_history,
        "last_answer": user_answer
    })

    current_question = followup_result["followup_question"]

    print(f"\nAI: {current_question}")


AI: Could you share a favorite pizza memory?

AI: What did you like most about the cheese?

AI: How did that stretchy, tasty cheese make your pizza even better?

AI: What made that combination so memorable?

AI: What made the taste so fantastic?

Interview completed 🎬
